This notebook encapsulates the pure Sentinel-1 processing chain.
This is the most involved part of the end-to-end workflow, and so is the most expensive to run.

Returns a datacube with the following bands:

- min_sse: the minimum sum of squared errors, applying the logistic curve as a window function
- min_sse_t: the time index of the minimum sum of squared errors (units: decimal years)
- sd: the standard deviation of the backscatter values (units: dB)
- p05: the 5th percentile of the backscatter values (units: dB)
- p95: the 95th percentile of the backscatter values (units: dB)

In [ ]:
import logging

import numpy as np
import openeo.processes
import shapely

import openeo

from utils import utils

logging.basicConfig(level=logging.INFO)

In [ ]:
connection = openeo.connect("openeo.dataspace.copernicus.eu")

In [ ]:
connection.authenticate_oidc()

In [ ]:
spatial_extent = {
    "west": 30.5503711040000994,
    "south": 1.0709279050000799,
    "east": 31.2229521229999989,
    "north": 1.5469373050000299,
}
temporal_extent = ["2019-10-01", "2022-04-01"]  # pad by ~3 months
band = "VH"
instrument_mode = "IW"
orbit_state = "ascending"
relative_orbit = 101

speckle_filter_radius = 2
speckle_filter_cv_noise = 1 / np.sqrt(4)
speckle_filter_temporal_window = 5

resample_spatial_resolution = 30  # m

logistic_window_size = 11
logistic_steepness_parameter = -2.0  # dimensionless

In [ ]:
# very small test AOI
x = 30.944
y = 1.273
delta = 0.1
spatial_extent = {
    "west": x,
    "south": y,
    "east": x + delta,
    "north": y + delta,
}

In [ ]:
# spatial extent as dict of Polygon geometry
spatial_extent = shapely.geometry.mapping(
    shapely.box(
        xmin=spatial_extent["west"],
        ymin=spatial_extent["south"],
        xmax=spatial_extent["east"],
        ymax=spatial_extent["north"],
    )
)

# Script

## Load S1 collection and apply sar_backscatter

In [ ]:
# collect outputs as we go, to built a multi-result process graph
process_graph_results = []

In [ ]:
# this collection has CRS = Auto42001
# which means that it will pick a UTM zone on the fly
s1_grd = connection.load_collection(
    "SENTINEL1_GRD",
    spatial_extent=spatial_extent,
    temporal_extent=temporal_extent,
    bands=[band],
    properties=[
        openeo.collection_property("sar:instrument_mode") == instrument_mode,
        openeo.collection_property("sat:orbit_state") == orbit_state,
        openeo.collection_property("sat:relative_orbit") == relative_orbit,
    ],
)

In [ ]:
process_graph_results.append(
    s1_grd.save_result(
        format="NetCDF",
        options={
            "filename_prefix": "0000_s1_grd",
        },
    )
)

In [ ]:
# convert from digital numbers to backscatter values
# https://open-eo.github.io/openeo-python-client/api.html#openeo.rest.datacube.DataCube.sar_backscatter

# Experimental openEO process
# Please note that this process is experimental with the potential for major things to change. Feel encouraged to try it out and give feedback, but refrain from using it in production.

# On this backend, only the following option is available:
# sigma0-ellipsoid: ground area computed with ellipsoid earth model

sigma_0 = s1_grd.sar_backscatter(
    coefficient="sigma0-ellipsoid",
    elevation_model="COPERNICUS_30",
)

In [ ]:
process_graph_results.append(
    sigma_0.save_result(
        format="NetCDF",
        options={
            "filename_prefix": "0010_sigma_0",
        },
    )
)

## speckle filters

In [ ]:
lee_udf = openeo.UDF.from_file(
    "../udf/lee_speckle_filter.py",
    runtime="Python",
    version="3.11",
    context={"radius": speckle_filter_radius, "cv_noise": speckle_filter_cv_noise},
)

In [ ]:
lee = sigma_0.apply_neighborhood(
    lee_udf,
    size=[
        {"dimension": "x", "value": 128, "unit": "px"},
        {"dimension": "y", "value": 128, "unit": "px"},
        # t: all (implicitly)
        # bands: all (implicitly)
    ],
    overlap=[
        {"dimension": "x", "value": speckle_filter_radius, "unit": "px"},
        {"dimension": "y", "value": speckle_filter_radius, "unit": "px"},
    ],
)

In [ ]:
process_graph_results.append(
    lee.save_result(
        format="NetCDF",
        options={
            "filename_prefix": "0020_lee",
        },
    )
)

In [ ]:
multitemporal_speckle_filter_udf = openeo.UDF.from_file(
    "../udf/multitemporal_speckle_filter_atbd.py",
    runtime="Python",
    version="3.11",
    context={
        "radius": speckle_filter_radius,
        "window_size": speckle_filter_temporal_window,
    },
)

In [ ]:
multitemporal = lee.apply_neighborhood(
    multitemporal_speckle_filter_udf,
    size=[
        {"dimension": "x", "value": 128, "unit": "px"},
        {"dimension": "y", "value": 128, "unit": "px"},
        # t: all (implicitly)
        # bands: all (implicitly)
    ],
    overlap=[
        {"dimension": "x", "value": speckle_filter_radius, "unit": "px"},
        {"dimension": "y", "value": speckle_filter_radius, "unit": "px"},
    ],
)

In [ ]:
process_graph_results.append(
    multitemporal.save_result(
        format="NetCDF",
        options={
            "filename_prefix": "0030_multitemporal",
        },
    )
)

## resample S1 to lower resolution

In [ ]:
lower_resolution = multitemporal.resample_spatial(
    resolution=resample_spatial_resolution,
    method="average",  # TODO: what is the correct method here?
)

In [ ]:
process_graph_results.append(
    lower_resolution.save_result(
        format="NetCDF",
        options={
            "filename_prefix": "0040_lower_resolution",
        },
    )
)

## convert to dB

In [ ]:
s1_dB: openeo.DataCube = utils.convert_to_dB(lower_resolution)

In [ ]:
process_graph_results.append(
    s1_dB.save_result(
        format="NetCDF",
        options={
            "filename_prefix": "0050_s1_dB",
        },
    )
)

# standard deviation

In [ ]:
standard_deviation = s1_dB.reduce_temporal(openeo.processes.sd)
standard_deviation = standard_deviation.rename_labels("bands", ["sd"])

In [ ]:
process_graph_results.append(
    standard_deviation.save_result(
        format="NetCDF",
        options={
            "filename_prefix": "0060_standard_deviation",
        },
    )
)
process_graph_results.append(
    standard_deviation.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0060_standard_deviation",
        },
    )
)

# percentiles

In [ ]:
def calculate_p05(data: openeo.processes.ProcessBuilder):
    return data.quantiles(probabilities=[0.05])


def calculate_p95(data: openeo.processes.ProcessBuilder):
    return data.quantiles(probabilities=[0.95])

In [ ]:
s1_dB_p05 = s1_dB.reduce_temporal(calculate_p05)
s1_dB_p95 = s1_dB.reduce_temporal(calculate_p95)

In [ ]:
s1_dB_p05 = s1_dB_p05.rename_labels("bands", ["p05"])
s1_dB_p95 = s1_dB_p95.rename_labels("bands", ["p95"])

In [ ]:
process_graph_results.append(
    s1_dB_p05.save_result(
        format="netCDF",
        options={
            "filename_prefix": "0070_s1_dB_p05",
        },
    )
)
process_graph_results.append(
    s1_dB_p05.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0070_s1_dB_p05",
        },
    )
)
process_graph_results.append(
    s1_dB_p95.save_result(
        format="netCDF",
        options={
            "filename_prefix": "0080_s1_dB_p95",
        },
    )
)
process_graph_results.append(
    s1_dB_p95.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0080_s1_dB_p95",
        },
    )
)

## logistic sum squared error

In [ ]:
logistic_udf = openeo.UDF.from_file(
    "../udf/logistic_curve_sse.py",
    runtime="Python",
    version="3.11",
    context={
        "window_size": logistic_window_size,
        "steepness_parameter": logistic_steepness_parameter,
    },
)

In [ ]:
logistic_sse = s1_dB.apply_dimension(
    process=logistic_udf,
    dimension="t",
)

In [ ]:
process_graph_results.append(
    logistic_sse.save_result(
        format="netCDF",
        options={
            "filename_prefix": "0090_logistic_sse",
        },
    )
)

In [ ]:
# reduce over time to find the best fit detection
min_sse = logistic_sse.reduce_temporal(openeo.processes.min)
min_sse = min_sse.rename_labels("bands", ["min_sse"])

In [ ]:
process_graph_results.append(
    min_sse.save_result(
        format="netCDF",
        options={
            "filename_prefix": "0100_min_sse",
        },
    )
)
process_graph_results.append(
    min_sse.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0100_min_sse",
        },
    )
)

In [ ]:
idxmin_t_udf = openeo.UDF.from_file(
    "../udf/idxmin_t.py",
    runtime="Python",
    version="3.11",
    context={},
)

In [ ]:
min_sse_t = logistic_sse.reduce_temporal(
    reducer=idxmin_t_udf,
)
min_sse_t = min_sse_t.rename_labels("bands", ["min_sse_t"])

In [ ]:
process_graph_results.append(
    min_sse_t.save_result(
        format="netCDF",
        options={
            "filename_prefix": "0110_min_sse_t",
        },
    )
)
process_graph_results.append(
    min_sse_t.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0110_min_sse_t",
        },
    )
)

## merge into single DataCube

In [ ]:
merged_cube = (
    standard_deviation.merge_cubes(s1_dB_p05)
    .merge_cubes(s1_dB_p95)
    .merge_cubes(min_sse)
    .merge_cubes(min_sse_t)
)

In [ ]:
process_graph_results.append(
    merged_cube.save_result(
        format="netCDF",
        options={
            "filename_prefix": "0120_merged_cube",
        },
    )
)
process_graph_results.append(
    merged_cube.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0120_merged_cube",
        },
    )
)

# Run batch job

In [ ]:
multi_result = openeo.MultiResult(process_graph_results)

In [ ]:
job = multi_result.create_job()
job.start_and_wait()
# Inspect job.logs() if it fails

In [ ]:
results = job.get_results()

In [ ]:
!mkdir -p output-script/
!rm -r output-script/

In [ ]:
results.download_files("output-script/")

In [ ]:
import json

with open("logs.json", "w") as f:
    json.dump(job.logs(), f, indent=2)